# Build Races Dimension

1. Read silver `races` table
1. Read silver `circuits` table
1. Join the data from `races` with `circuits` using `circuit_id`
1. Select the required columns
    - races.season 
    - races.round 
    - races.race_name 
    - races.race_date 
    - circuits.circuit_name 
    - circuits.locality 
    - circuits.country
1. Write the transformed data to gold `dim_races` table

> Below changes are required to implement Incremental Load Processing
1. Accept batch_id as a parameter to the notebook
1. Process data for only the batch_id being passed in (i.e., filter reading from silver using the batch_id)
1. Add created_timestamp, updated_timestamp to the gold table. 
1. Merge the processed data to the gold table
    - created_timestamp should only be populated at the time of inserting/ creating the record. It should not be updated during the merge update.

### Step 1 - Loading the env config from the common config

In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
%run ../00-common-Config/02-helper-function

In [0]:
import pyspark.sql.functions as f

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
target_table=f"{catalog_name}.{gold_schema}.dim_race"


In [0]:
circuit_df=(
    spark.table(f"{catalog_name}.{silver_schema}.circuits")
    .filter(f.col("batch_id")==v_batch_id)
)
race_df=(
    spark.table(f"{catalog_name}.{silver_schema}.races")
    .filter(f.col("batch_id")==v_batch_id)
)

In [0]:
dim_race_final_df = race_df.join(
    circuit_df, race_df.circuit_id == circuit_df.circuit_id, "inner"
).select(
    race_df.race_name,
    race_df.race_date,
    race_df.season,
    race_df.round,
    circuit_df.circuit_name,
    circuit_df.country,
    circuit_df.locality,
)

In [0]:

# (
#     dim_race_final_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
# )
write_to_gold(
    input_df=dim_race_final_df,
    target_table=target_table,
    merge_condition="t.season = s.season AND t.round = s.round",
    columns_to_update=[
        "race_name",
        "race_date",
        "circuit_name",
        "locality",
        "country"
    ]
)

In [0]:
display(spark.table(target_table))